# RAGAS Evaluation

End-to-end RAG evaluation using [RAGAS](https://docs.ragas.io/) metrics
against a hand-curated set of 20 PyTorch questions.

## 1. Environment Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import logging
import os
import sys
import textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-25s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("notebook")

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_KEY      = os.environ["GROQ_API_KEY"]
QDRANT_URL    = os.environ["QDRANT_URL"]
QDRANT_KEY    = os.environ["QDRANT_API_KEY"]
HF_TOKEN      = os.environ.get("HUGGINGFACEHUB_API_TOKEN", "")
EMBEDDER_MODE = os.environ.get("EMBEDDER_MODE", "auto")
COLLECTION    = os.environ.get("COLLECTION_NAME", "pytorch_docs")

TOP_K       = 6
TEMPERATURE = 0.0
MAX_TOKENS  = 1024

EVAL_DIR = PROJECT_ROOT / "data" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE    = EVAL_DIR / "reference_answers.json"
RAG_RESULTS_FILE  = EVAL_DIR / "rag_results.json"
RAGAS_SCORES_FILE = EVAL_DIR / "ragas_scores.csv"
CHECKPOINT_FILE   = EVAL_DIR / "ragas_checkpoint.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Eval dir     : {EVAL_DIR}")
print(f"Embedder mode: {EMBEDDER_MODE}")
print(f"Collection   : {COLLECTION}")

## 2. Evaluation Question Set

Twenty questions covering:
- **Tensor basics** (creation, device, dtype)
- **Autograd & gradient mechanics**
- **`torch.nn` modules and training-loop idioms**
- **Performance primitives** (`autocast`, `compile`, profiler)
- **Data loading** and **distributed training** concepts

In [ ]:
EVAL_QUESTIONS: list[str] = [
    # Tensor basics
    "How do I move a tensor to GPU?",
    "What is the difference between torch.Tensor and torch.tensor?",
    "What does tensor.detach() do and how does it differ from torch.no_grad()?",
    "How does torch.einsum work? Provide a matrix-multiplication example.",
    # Autograd & gradient mechanics
    "What does torch.no_grad() do and when should I use it?",
    "What is the purpose of optimizer.zero_grad()?",
    "How do I implement gradient clipping in PyTorch?",
    "What is torch.autograd.Function and when would I subclass it?",
    # Model definition
    "What is the difference between torch.nn.BatchNorm1d and torch.nn.LayerNorm?",
    "What is a torch.nn.ModuleList and when should I use it instead of a Python list?",
    "What is the difference between model.eval() and model.train()?",
    # Training-loop idioms
    "How do I save and load a model checkpoint?",
    "How do I freeze model parameters during fine-tuning?",
    "How does gradient checkpointing reduce memory usage?",
    "How do I implement a custom learning-rate scheduler in PyTorch?",
    # Performance & mixed precision
    "How does torch.autocast work in mixed-precision training?",
    "What is torch.compile() and what benefits does it offer over eager mode?",
    "How do I profile a PyTorch model to find performance bottlenecks?",
    # Data loading & distributed
    "How do I use torch.utils.data.DataLoader with a custom Dataset?",
    "How does torch.nn.DataParallel differ from DistributedDataParallel?",
]

assert len(EVAL_QUESTIONS) == 3

print(f"{len(EVAL_QUESTIONS)} evaluation questions:")
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"  Q{i:02d}. {q}")

## 3. Reference Answers

RAGAS metrics `ContextRecall`, `ContextPrecision`, and `AnswerCorrectness`
require a ground-truth reference answer per question.

We generate them with **Claude Sonnet 4.6** on claude.ai.

### Prompt to generate the reference answers

In [ ]:
question_block = "\n".join(
    f"Q{i:02d}: {q}" for i, q in enumerate(EVAL_QUESTIONS, 1)
)

CLAUDE_PROMPT = (
    "You are an expert PyTorch engineer with thorough knowledge of the official\n"
    "PyTorch documentation.\n"
    "\n"
    "For each numbered question below, write a concise and technically accurate\n"
    "reference answer (3-5 sentences). Rules:\n"
    "- Name specific API symbols, parameters, and module paths where relevant.\n"
    "- Do not fabricate; if something is version-dependent, say so.\n"
    "- Do not include code blocks — prose only.\n"
    "\n"
    "Respond ONLY with a valid JSON object structured exactly as:\n"
    "{\n"
    '  \"Q01\": \"answer for question 1 ...\",\n'
    '  \"Q02\": \"answer for question 2 ...\",\n'
    "  ...\n"
    "}\n"
    "Output raw JSON only — no preamble, no markdown fences, no trailing commentary.\n"
    "\n"
    "Questions:\n"
    + question_block
)

print("=" * 72)
print()
print(CLAUDE_PROMPT)
print()
print("=" * 72)

### Reference model response

In [ ]:
CLAUDE_JSON_RESPONSE = """
{
  "Q01": "Call tensor.to('cuda') or tensor.cuda() to move a tensor to the default GPU, or specify a device index with tensor.to('cuda:0'). You can also construct a torch.device object and pass it to .to(). The operation returns a new tensor on the target device; the original is not modified in place. For portability, prefer tensor.to(device) where device is determined at runtime via torch.device('cuda' if torch.cuda.is_available() else 'cpu').",
  "Q02": "torch.Tensor is the Python class (a type alias for torch.FloatTensor by default), so calling it as a constructor with shape arguments allocates uninitialized memory of the default dtype. torch.tensor is a factory function that copies data from an existing array-like object, infers the dtype from the data, and always creates a leaf tensor with requires_grad=False by default. Use torch.tensor when you have concrete data; prefer torch.empty, torch.zeros, or torch.ones when you only need a shape.",
  "Q03": "tensor.detach() returns a new tensor that shares the same storage but is detached from the computation graph, meaning no gradients will flow through it during backpropagation; requires_grad is False on the result. torch.no_grad() is a context manager (or decorator) that disables gradient tracking for all operations within its scope without creating detached copies. Use detach() when you want to reuse a tensor's data as a constant within a larger graph; use no_grad() for inference loops where you want to suppress the overhead of building the graph entirely.",
  "Q04": "torch.einsum(equation, *operands) performs tensor contractions described by an Einstein summation string, where each letter denotes a dimension and repeated letters imply summation. For matrix multiplication of A (shape m×k) and B (shape k×n), the call is torch.einsum('mk,kn->mn', A, B), where the shared index k is contracted (summed) and the output retains m and n. The function supports broadcasting, batched operations, and arbitrary multi-tensor contractions beyond what torch.matmul exposes.",
  "Q05": "torch.no_grad() is a context manager that disables the autograd engine for all tensor operations executed within its block, preventing PyTorch from building the computation graph and storing intermediate activations. This reduces memory consumption and speeds up computation, making it appropriate for inference, evaluation loops, and any code path where gradients are not needed. It can also be used as a function decorator. Unlike tensor.detach(), it does not produce new tensor objects; it simply suppresses graph construction globally within its scope.",
  "Q06": "optimizer.zero_grad() sets the .grad attribute of every parameter the optimizer manages to zero (or None when set_to_none=True, the default since PyTorch 1.7). This is necessary because PyTorch accumulates gradients into .grad by addition every time loss.backward() is called, so without zeroing, gradients from successive iterations would sum together. Calling it at the start of each training step (before the forward pass) is the standard pattern, though some advanced techniques deliberately skip zeroing to implement gradient accumulation.",
  "Q07": "The primary API is torch.nn.utils.clip_grad_norm_(parameters, max_norm, norm_type=2.0), which rescales all gradients collectively so their global norm does not exceed max_norm; it returns the total norm before clipping. An alternative is torch.nn.utils.clip_grad_value_(parameters, clip_value), which clips each gradient element independently to the range [-clip_value, clip_value]. Both functions must be called after loss.backward() and before optimizer.step(), and they accept either a model's parameters() iterator or a list of tensors.",
  "Q08": "torch.autograd.Function lets you define custom differentiable operations by subclassing it and implementing static forward() and backward() methods, giving you full control over both the computation and its gradient formula. You register the operation's context (saved tensors, flags) via ctx.save_for_backward() in forward and retrieve them in backward to compute input gradients. This is necessary when a mathematical operation has no native PyTorch equivalent, when you need to interface with a non-PyTorch library (e.g., a CUDA kernel), or when the default autograd-derived gradient is numerically unstable and you want to supply an analytically stable one.",
  "Q09": "torch.nn.BatchNorm1d normalizes over the batch and spatial dimensions (N and L), computing per-channel statistics (mean and variance) that depend on the entire mini-batch at training time and on running statistics at eval time; it is therefore batch-size-dependent and behaves differently in model.train() versus model.eval(). torch.nn.LayerNorm normalizes over a specified normalized_shape (typically the feature dimensions of a single sample), so its statistics are computed independently per sample and the behavior is identical at train and eval time. BatchNorm is preferred in CNNs with large batches, while LayerNorm is standard in transformers and settings with small or variable batch sizes.",
  "Q10": "torch.nn.ModuleList is a container that holds nn.Module instances in a list and registers them properly with the parent module, so their parameters appear in model.parameters(), are moved by .to(device), and are saved by state_dict(). A plain Python list does not register its contents with PyTorch's module system, causing those parameters to be invisible to the optimizer and missing from checkpoints. Use ModuleList when you have a dynamic or data-driven number of layers, or when you need to index into layers individually rather than calling them sequentially as in nn.Sequential.",
  "Q11": "model.train() and model.eval() set a mode flag that affects modules whose behavior differs between training and inference, most notably torch.nn.Dropout (which is active only during training) and torch.nn.BatchNorm layers (which use batch statistics during training and running statistics during eval). They do not affect the computation graph or gradient computation directly; you still need torch.no_grad() separately to suppress autograd during eval. Both methods are recursive and set the mode on all child modules; they return the model itself to allow chaining.",
  "Q12": "The recommended approach is to save only the state dictionary with torch.save(model.state_dict(), 'checkpoint.pt') and reload it with model.load_state_dict(torch.load('checkpoint.pt', map_location=device)). For full training resumption, also save the optimizer state, epoch, and any scheduler state in a single dict passed to torch.save. Avoid pickling the entire model object (torch.save(model, ...)) because it couples the checkpoint to the exact source code structure and is fragile across refactors.",
  "Q13": "Set requires_grad=False on the parameters you want to freeze, either individually or via model.parameters() filtered by name. A convenience method is to call param.requires_grad_(False) on a submodule's parameters, or equivalently freeze the whole submodule with for param in submodule.parameters(): param.requires_grad = False. Frozen parameters will not receive gradient updates and consume no gradient memory, but you must pass only the unfrozen parameters to the optimizer (e.g., optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)) so the optimizer does not attempt to update them.",
  "Q14": "Gradient checkpointing, exposed via torch.utils.checkpoint.checkpoint(function, *inputs), trades compute for memory by not storing intermediate activations during the forward pass; instead, the selected segment is recomputed from its inputs during the backward pass when gradients are needed. This can reduce activation memory by roughly the square root of the network depth, at the cost of one additional forward pass through the checkpointed segments. It is particularly useful for very deep networks or large sequence models (e.g., transformers with long contexts) where activation memory is the binding constraint.",
  "Q15": "Subclass torch.optim.lr_scheduler.LRScheduler (called _LRScheduler before PyTorch 2.0) and override the get_lr() method, which must return a list of learning rates—one per optimizer param group—computed from self.last_epoch and self.base_lrs. Attach the scheduler to an existing optimizer by passing it as the first constructor argument, then call scheduler.step() once per epoch (or per step, depending on your design) after optimizer.step(). Alternatively, torch.optim.lr_scheduler.LambdaLR accepts a plain function mapping the epoch index to a multiplicative factor, which is often sufficient for simple custom schedules.",
  "Q16": "torch.autocast (used as a context manager, e.g., with torch.autocast(device_type='cuda', dtype=torch.float16)) automatically casts eligible operations to a lower-precision dtype (float16 or bfloat16) while keeping numerically sensitive operations (reductions, softmax, loss computation) in float32, according to a built-in op-level policy. It is typically paired with torch.cuda.amp.GradScaler, which scales the loss before backward() to prevent float16 underflow and unscales gradients before the optimizer step. Together they constitute PyTorch's Automatic Mixed Precision (AMP) workflow, reducing memory and increasing throughput on Tensor Core–capable GPUs with minimal code change.",
  "Q17": "torch.compile(model), introduced in PyTorch 2.0, applies TorchDynamo to capture the model's computation graph via Python bytecode analysis and then lowers it through TorchInductor (by default) to generate optimized Triton or C++ kernels. Benefits over eager mode include operator fusion, reduced Python overhead, and better hardware utilization, often yielding 1.5–3× speedups on GPUs without requiring code changes beyond the compile call. The backend parameter (e.g., 'inductor', 'cudagraphs', 'onnxrt') and mode parameter ('default', 'reduce-overhead', 'max-autotune') let you trade compilation time against runtime performance.",
  "Q18": "The primary tool is torch.profiler.profile, a context manager that records CPU and CUDA kernel timings, memory allocation, and optionally stack traces. Wrap the training loop with it, use schedule=torch.profiler.schedule(wait, warmup, active) to control which steps are profiled, and export results with prof.export_chrome_trace('trace.json') for visualization in chrome://tracing or TensorBoard via prof.export_stacks. For lighter-weight, line-level bottleneck analysis, torch.autograd.profiler.profile is an older alternative, while NVIDIA Nsight Systems or nvprof provide deeper GPU kernel-level insight outside PyTorch.",
  "Q19": "Subclass torch.utils.data.Dataset and implement __len__() (returning the dataset size) and __getitem__(idx) (returning a single sample). Pass an instance of your dataset to torch.utils.data.DataLoader along with parameters such as batch_size, shuffle, num_workers (for multiprocess loading), collate_fn (to customize batch assembly), and pin_memory=True (to accelerate host-to-GPU transfers). The DataLoader handles batching, shuffling, and parallel prefetching automatically; for map-style datasets with non-sequential access patterns, you can also supply a custom Sampler or BatchSampler.",
  "Q20": "torch.nn.DataParallel (DDP-lite) runs on a single process and replicates the model across multiple GPUs on one machine, scattering each batch across devices and gathering outputs on the primary GPU; this design creates a GPU utilization imbalance and a Python GIL bottleneck that limits scalability. torch.nn.parallel.DistributedDataParallel launches one process per GPU (via torch.distributed), keeps a full model replica on each, and synchronizes gradients with an all-reduce collective after backward(), eliminating the gather bottleneck and scaling efficiently across multiple nodes. DDP is the recommended approach for any serious multi-GPU training; DataParallel is retained mainly for quick experiments on a single machine."
}
"""

### Validate, preview, and persist

In [ ]:
reference_answers: dict[str, str] = {}

if CLAUDE_JSON_RESPONSE and "PASTE CLAUDE" not in CLAUDE_JSON_RESPONSE:
    try:
        raw: dict[str, str] = json.loads(CLAUDE_JSON_RESPONSE.strip())
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Claude's response is not valid JSON: {exc}\n"
            "Make sure you copied the complete JSON block without markdown fences."
        ) from exc

    missing = [f"Q{i:02d}" for i in range(1, 21) if f"Q{i:02d}" not in raw]
    if missing:
        raise KeyError(
            f"Keys missing in Claude's response: {missing}\n"
        )

    reference_answers = {
        EVAL_QUESTIONS[i - 1]: raw[f"Q{i:02d}"]
        for i in range(1, len(EVAL_QUESTIONS) + 1)
    }
    with REFERENCE_FILE.open("w", encoding="utf-8") as fh:
        json.dump(reference_answers, fh, indent=2, ensure_ascii=False)
    print(f">> Saved {len(reference_answers)} reference answers: {REFERENCE_FILE}")

# Load from disk if available
elif REFERENCE_FILE.exists():
    with REFERENCE_FILE.open(encoding="utf-8") as fh:
        reference_answers = json.load(fh)
    print(f">> Loaded {len(reference_answers)} reference answers from {REFERENCE_FILE}")

else:
    raise RuntimeError(
        "No reference answers found.\n"
        "Complete §3.2 first: paste Claude's JSON response and re-run."
    )

print()

q, a = list(reference_answers.items())[0]
print(f"Q: {q}")
print(f"A: {a}")
print()

## 4. Run RAG Chain

### 4.1 — Build the pipeline

In [ ]:
import torch
from qdrant_client import QdrantClient

from src.rag.chain import build_rag_chain
from src.retrieval import HyDETransformer
from src.vectorstore import QdrantDocStore


def _build_embedder():
    """
    Instantiate the appropriate embedder, mirroring ``bot/services.py`` logic.

    Tries local BGEM3Embedder first when EMBEDDER_MODE is 'auto' or 'local',
    then falls back to HFInferenceEmbedder if the local model isn't available
    (no GPU, missing FlagEmbedding dependency, etc.).
    """
    mode = EMBEDDER_MODE.strip().lower()
    if mode in ("local", "auto"):
        try:
            from src.embedding import BGEM3Embedder
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"Loading local BGEM3Embedder on {device} ...")
            emb = BGEM3Embedder(batch_size=1)
            print("Local embedder ready.")
            return emb, "local"
        except (ImportError, Exception) as exc:
            if mode == "local":
                raise
            print(f"Local embedder unavailable ({exc}), falling back to HF Inference.")
    if not HF_TOKEN:
        raise RuntimeError(
            "HF Inference fallback requires HUGGINGFACEHUB_API_TOKEN in .env"
        )
    from src.embedding.hf_embedder import HFInferenceEmbedder
    print("Using HFInferenceEmbedder ...")
    return HFInferenceEmbedder(api_token=HF_TOKEN), "hf"


embedder, embedder_kind = _build_embedder()

qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY)
store = QdrantDocStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedder=embedder,
)
info = store.collection_info()
print(f"Collection '{info['name']}': {info['points_count']:,} points, status={info['status']}")

hyde = HyDETransformer(groq_api_key=GROQ_KEY)
retriever = store.as_retriever(top_k=TOP_K, hyde=hyde)

chain = build_rag_chain(
    retriever=retriever,
    groq_api_key=GROQ_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)
print("RAG chain ready.")

### 4.2 — Run all 20 questions

In [ ]:
# Load existing results from disk
if RAG_RESULTS_FILE.exists():
    with RAG_RESULTS_FILE.open(encoding="utf-8") as fh:
        rag_results: list[dict] = json.load(fh)
    print(f"Loaded {len(rag_results)} existing RAG results from {RAG_RESULTS_FILE}")
else:
    rag_results = []

# Find questions not yet answered
_NA_VALUES = {None, "", "N/A"}

answered = {
    r["question"]
    for r in rag_results
    if r.get("answer") not in _NA_VALUES
}
todo = [
    (i, q) for i, q in enumerate(EVAL_QUESTIONS, 1)
    if q not in answered
]

# Report what was loaded and what still needs work
na_cached = [r["question_id"] for r in rag_results if r.get("answer") in _NA_VALUES]
if na_cached:
    print(f"{len(na_cached)} cached result(s) are N/A and will be re-evaluated: "
          f"{na_cached}")

if not todo:
    print("All questions already answered — skipping RAG chain.")
else:
    print(f"Running RAG chain for {len(todo)} question(s) "
          f"({len(answered)} already cached) ...")
    for i, question in tqdm(todo, desc="Running RAG chain"):
        result: RAGResult = chain.invoke(question)
        # Remove the stale N/A entry for this question before appending the fresh one
        rag_results = [r for r in rag_results if r["question"] != question]
        rag_results.append({
            "question_id":  f"Q{i:02d}",
            "question":     question,
            "answer":       result.answer,
            "contexts":     [doc.page_content for doc in result.context_docs],
            "context_urls": [
                doc.metadata.get("citation_url", "") for doc in result.context_docs
            ],
            "sources": [
                {"index": s.index, "url": s.url, "title": s.title, "symbol": s.symbol}
                for s in result.sources
            ],
            "reference": reference_answers.get(question, ""),
        })

    # Keep results ordered by question index regardless of insertion order
    question_order = {q: i for i, q in enumerate(EVAL_QUESTIONS)}
    rag_results.sort(key=lambda r: question_order[r["question"]])

    with RAG_RESULTS_FILE.open("w", encoding="utf-8") as fh:
        json.dump(rag_results, fh, indent=2, ensure_ascii=False)
    print(f"\n✓ Saved {len(rag_results)} RAG results → {RAG_RESULTS_FILE}")

## 5. Build RAGAS Dataset

| `SingleTurnSample` field | Source |
|---|---|
| `user_input` | `question` |
| `retrieved_contexts` | list of retrieved chunk texts |
| `response` | LLaMA-generated answer |
| `reference` | Claude Sonnet 4.6 reference answer |

In [ ]:
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample

# Reload from disk when re-running this section standalone
if not rag_results and RAG_RESULTS_FILE.exists():
    with RAG_RESULTS_FILE.open(encoding="utf-8") as fh:
        rag_results = json.load(fh)
    print(f"Loaded {len(rag_results)} RAG results from {RAG_RESULTS_FILE}")

samples = [
    SingleTurnSample(
        user_input=r["question"],
        retrieved_contexts=r["contexts"],
        response=r["answer"],
        reference=r["reference"],
    )
    for r in rag_results
]
eval_dataset = EvaluationDataset(samples=samples)

print(f"EvaluationDataset: {len(eval_dataset)} samples")
print()
s0 = samples[0]
print(f"user_input         : {s0.user_input}")
print(f"retrieved_contexts : {len(s0.retrieved_contexts)} chunks")
print(f"response  (preview): {s0.response[:50]}...")
print(f"reference (preview): {s0.reference[:50]}...")

## 6. Configure RAGAS Evaluator

- **Judge LLM**: `llama-3.3-70b-versatile` via Groq (same as the chain)
- **Embeddings**: `ProjectEmbeddings` adapter from `src.embedding.lc_adapter`

### Judge LLM

In [ ]:
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
 
_JUDGE_MAX_TOKENS = 10000   # ← was effectively 3 072 with llm_factory default
 
judge_llm = LangchainLLMWrapper(
    ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_KEY,
        temperature=0.0,
        max_tokens=_JUDGE_MAX_TOKENS,
    )
)
print(f"Judge LLM ready  (max_tokens={_JUDGE_MAX_TOKENS})")

### Embeddings adapter

`ProjectEmbeddings` (in `src/embedding/lc_adapter.py`) is a thin
`langchain_core.embeddings.Embeddings` wrapper around the project embedder.
Both `BGEM3Embedder` and `HFInferenceEmbedder` share the same
`embed(texts) -> np.ndarray` interface, so the adapter works with either
backend transparently.

In [ ]:
import asyncio
from typing import List
 
from langchain_core.embeddings import Embeddings as LCEmbeddings
from ragas.embeddings import BaseRagasEmbedding
 
 
class ProjectRagasEmbeddings(BaseRagasEmbedding, LCEmbeddings):
    def __init__(self, embedder):
        self._emb = embedder
 
    # ── LangChain interface (called by RAGAS internally) ──────────────────
    def embed_query(self, text: str) -> List[float]:
        return self._emb.embed([text])[0].tolist()
 
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._emb.embed(texts).tolist()
 
    # ── RAGAS interface ───────────────────────────────────────────────────
    def embed_text(self, text: str) -> List[float]:
        return self.embed_query(text)
 
    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)
 
    async def aembed_text(self, text: str) -> List[float]:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self.embed_query, text)
 
    async def aembed_texts(self, texts: List[str]) -> List[List[float]]:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self.embed_documents, texts)
 
 
ragas_embeddings = ProjectRagasEmbeddings(embedder)
print(f"Embeddings adapter ready (backend: {embedder_kind})")

## 7. Run Evaluation

| Metric | Needs reference | Needs embeddings | Measures |
|---|---|---|---|
| `Faithfulness` | No | No | Is the answer grounded in the retrieved context? |
| `AnswerRelevancy` | No | Yes | Is the answer on-topic for the question? |
| `ContextPrecision` | Yes | No | Are retrieved chunks useful for the reference? |
| `ContextRecall` | Yes | No | Does the context cover the reference's claims? |
| `AnswerCorrectness` | Yes | Yes | How correct is the answer vs the reference? |

In [ ]:
from src.evaluation import build_metrics, run_evaluation

metrics    = build_metrics(judge_llm, ragas_embeddings)
checkpoint = await run_evaluation(samples, metrics, CHECKPOINT_FILE)

## 8. Results

### 8.1 — Per-question scores

In [ ]:
import math

# Load checkpoint from disk if available
if "checkpoint" not in dir() or not checkpoint:
    with CHECKPOINT_FILE.open(encoding="utf-8") as fh:
        checkpoint = json.load(fh)
    print(f"Loaded checkpoint from {CHECKPOINT_FILE}")

rows = []
for i, question in enumerate(EVAL_QUESTIONS, 1):
    qid = f"Q{i:02d}"
    short_q = question[:55] + "..." if len(question) > 55 else question
    scores = checkpoint.get(qid) or {}
    row = {"id": qid, "question_short": short_q, **scores}
    rows.append(row)

scores_df = pd.DataFrame(rows)
metric_cols = [c for c in scores_df.columns if c not in ("id", "question_short")]

# Warn about any samples that are still missing or have null scores
missing = [r["id"] for r in rows if not checkpoint.get(r["id"])]
nulls   = [
    r["id"] for r in rows
    if checkpoint.get(r["id"]) and any(v is None for v in checkpoint[r["id"]].values())
]
if missing:
    print(f"Not yet scored: {missing} — run §7 to complete.")
if nulls:
    print(f"Partial scores (some metrics null): {nulls} — re-run §7 to retry.")

# Persist
scores_df.to_csv(RAGAS_SCORES_FILE, index=False)
print(f"Scores saved → {RAGAS_SCORES_FILE}")
print()

display(
    scores_df.style
        .format({c: lambda v: "N/A" if v is None or (isinstance(v, float) and math.isnan(v)) else f"{v:.3f}" for c in metric_cols})
        .background_gradient(subset=metric_cols, cmap="RdYlGn", vmin=0, vmax=1)
        .set_caption("RAGAS per-question scores  (0 = worst, 1 = best)")
        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

### 8.2 — Summary statistics

In [ ]:
summary = scores_df[metric_cols].agg(["mean", "median", "std", "min", "max"])

print()
print("RAGAS Summary  (n=20 questions)")
print()
display(
    summary.style
        .format("{:.3f}")
        .background_gradient(subset=metric_cols, cmap="RdYlGn", vmin=0, vmax=1)
)

# Copy-pasteable README badge line
headline_parts = [
    f"{col.replace('_', ' ').title()} {summary.loc['mean', col]:.2f}"
    for col in metric_cols
]
print()
print("README badge line:")
print("  RAGAS (n=20, judge=LLaMA-3.3-70B): " + " | ".join(headline_parts))

### 8.3 — Weakest samples (bottom 5 by mean score)

These are the questions to focus retrieval and prompt-tuning effort on.

In [ ]:
scores_df["mean_score"] = scores_df[metric_cols].mean(axis=1)
worst = scores_df.nsmallest(5, "mean_score")[
    ["id", "question_short", "mean_score"] + metric_cols
]

print("Bottom 5 questions by mean RAGAS score:")
display(
    worst.style
        .format({c: "{:.3f}" for c in metric_cols + ["mean_score"]})
        .background_gradient(
            subset=metric_cols + ["mean_score"], cmap="RdYlGn", vmin=0, vmax=1
        )
)

## Single question inspection

Change `INSPECT_ID` to any question ID (`Q01`–`Q20`) to see the full answer, reference, retrieved URLs, and score breakdown side by side.

In [ ]:
INSPECT_ID = "Q12"

idx = int(INSPECT_ID[1:]) - 1
rec = rag_results[idx]
row = scores_df[scores_df["id"] == INSPECT_ID].iloc[0]

print(f">> Inspection: {INSPECT_ID}")

print(f"\nQuestion:\n  {rec['question']}")

print("\nReference answer (Claude Sonnet 4.6):")
for line in textwrap.wrap(rec["reference"], 70):
    print(f"  {line}")

print("\nRAG answer (LLaMA-3.3-70B):")
for line in textwrap.wrap(rec["answer"], 70):
    print(f"  {line}")

print("\nRetrieved context URLs:")
for url in rec["context_urls"]:
    print(f"  {url}")

print("\nRAGAS scores:")
for col in metric_cols:
    val = float(row.get(col, float("nan")))
    filled = int(val * 20)
    bar = "█" * filled + "░" * (20 - filled)
    print(f"  {col:<28} {val:.3f}  [{bar}]")